# CardioIA – Fase 2
## Parte 1 – Extração de Sintomas e Sugestão de Diagnóstico

Este notebook foi ajustado para funcionar localmente e no Google Colab, localizando automaticamente a pasta `data`.

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
def localizar_base():
    candidatos = [
        Path('.'),
        Path('cardioia-fase2-diagnostico-automatizado'),
        Path('/content/cardioia-fase2-diagnostico-automatizado'),
    ]
    for candidato in candidatos:
        if (candidato / 'data').exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível localizar a pasta 'data'. Se estiver no Colab, clone o repositório primeiro."
    )

base = localizar_base()
print('Base encontrada em:', base.resolve())

In [ ]:
frases_path = base / 'data' / 'sintomas_pacientes.txt'
mapa_path = base / 'data' / 'mapa_conhecimento_sintomas_doencas.csv'

with open(frases_path, 'r', encoding='utf-8') as f:
    frases = [linha.strip() for linha in f.readlines() if linha.strip()]

mapa = pd.read_csv(mapa_path)
print('Quantidade de frases:', len(frases))
display(mapa.head())

In [ ]:
def identificar_sintomas(frase, mapa):
    frase_lower = frase.lower()
    sintomas = []
    doencas = []

    for _, row in mapa.iterrows():
        s1 = str(row['sintoma_1']).lower()
        s2 = str(row['sintoma_2']).lower()
        doenca = row['doenca_associada']

        if s1 in frase_lower:
            sintomas.append(row['sintoma_1'])
            doencas.append(doenca)
        if s2 in frase_lower:
            sintomas.append(row['sintoma_2'])
            doencas.append(doenca)

    diagnostico = pd.Series(doencas).value_counts().idxmax() if doencas else 'Sem diagnóstico'
    return sintomas, diagnostico

In [ ]:
resultados = []
for frase in frases:
    sintomas, diagnostico = identificar_sintomas(frase, mapa)
    resultados.append({
        'frase': frase,
        'sintomas_encontrados': ', '.join(sintomas) if sintomas else 'Nenhum',
        'diagnostico_sugerido': diagnostico
    })

resultado_df = pd.DataFrame(resultados)
display(resultado_df)

### Se estiver no Google Colab
Execute antes:

```python
!git clone https://github.com/limadeivisson/cardioia-fase2-diagnostico-automatizado.git
```